# Real vs Synthetic CAM Similarity

            对同一 test sample，比对 `real_train` 和 A/B/C synthetic-trained evaluator 的 CAM 是否关注同一区域。

            指标：

            - Pearson correlation
            - Cosine similarity
            - Top-10% hot region IoU
            - Top-10% hot region Dice


In [ ]:

from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

EXP_ROOT = Path("/data/zengqiang/experiments/ncfm_medmnist_ablation_20260519")

def require_exp_root():
    if not EXP_ROOT.exists():
        raise FileNotFoundError(
            f"EXP_ROOT not found: {EXP_ROOT}. "
            "Edit EXP_ROOT in the first code cell to your experiment directory."
        )

def ensure_report_dir(*parts):
    path = EXP_ROOT / "reports" / "cam" / Path(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_group(name):
    if name == "real_train":
        return "real_train"
    if name.startswith("ipc10_"):
        return name[len("ipc10_"):]
    return name

def resolve_result_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if str(value).startswith("/"):
        return p
    q = EXP_ROOT / value
    return q

def load_eval_metrics():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "runs").glob("*/ipc10/*/eval_metrics_best.json")):
        item = read_json(path)
        item["dataset"] = path.parents[2].name
        item["group"] = path.parent.name
        item["metrics_path"] = str(path)
        rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"A_pure_ncfd_wopsi": 0, "B_minmax_ncfm_psi": 1, "C_code_default_enhanced": 2}
    df["_order"] = df["group"].map(order).fillna(99)
    return df.sort_values(["dataset", "_order"]).drop(columns=["_order"])

def load_cam_summaries():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "results" / "cam").glob("*/*/summary.csv")):
        dataset = path.parents[1].name
        group = normalize_group(path.parent.name)
        df = pd.read_csv(path)
        if df.empty:
            continue
        df["dataset"] = dataset
        df["group"] = group
        df["summary_path"] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    for col in ["index", "y_true", "y_pred", "confidence", "correct", "cam_entropy", "topk_activation_ratio"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def cam_group_summary(cam_df):
    if cam_df.empty:
        return cam_df
    grouped = (
        cam_df.groupby(["dataset", "group"], as_index=False)
        .agg(
            n=("index", "count"),
            cam_acc=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_entropy=("cam_entropy", "mean"),
            mean_top10_mass=("topk_activation_ratio", "mean"),
            correct_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
            correct_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
        )
    )
    order = {"real_train": 0, "A_pure_ncfd_wopsi": 1, "B_minmax_ncfm_psi": 2, "C_code_default_enhanced": 3}
    grouped["_order"] = grouped["group"].map(order).fillna(99)
    return grouped.sort_values(["dataset", "_order"]).drop(columns=["_order"])


In [ ]:

cam_df = load_cam_summaries()
if cam_df.empty:
    raise RuntimeError("No CAM summaries found.")

def read_cam(path_value):
    path = resolve_result_path(path_value)
    if not path.exists():
        return None
    img = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0
    cam = img[..., 0]
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-12)
    return cam

def sim_metrics(a, b, top_fraction=0.10):
    a = a.reshape(-1).astype(np.float64)
    b = b.reshape(-1).astype(np.float64)
    pearson = np.corrcoef(a, b)[0, 1] if np.std(a) > 1e-12 and np.std(b) > 1e-12 else np.nan
    cosine = float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-12))
    k = max(1, int(round(a.size * top_fraction)))
    a_hot = np.zeros_like(a, dtype=bool)
    b_hot = np.zeros_like(b, dtype=bool)
    a_hot[np.argsort(a)[-k:]] = True
    b_hot[np.argsort(b)[-k:]] = True
    inter = np.logical_and(a_hot, b_hot).sum()
    union = np.logical_or(a_hot, b_hot).sum()
    iou = float(inter / max(union, 1))
    dice = float(2 * inter / max(a_hot.sum() + b_hot.sum(), 1))
    return {"pearson": pearson, "cosine": cosine, "top10_iou": iou, "top10_dice": dice}

rows = []
for dataset, ddf in cam_df.groupby("dataset"):
    real = ddf[ddf["group"] == "real_train"].set_index("index")
    if real.empty:
        continue
    for group, gdf in ddf[ddf["group"] != "real_train"].groupby("group"):
        syn = gdf.set_index("index")
        for idx in sorted(set(real.index) & set(syn.index)):
            real_cam = read_cam(real.loc[idx, "cam_path"])
            syn_cam = read_cam(syn.loc[idx, "cam_path"])
            if real_cam is None or syn_cam is None:
                continue
            item = {
                "dataset": dataset,
                "group": group,
                "index": int(idx),
                "real_correct": int(real.loc[idx, "correct"]),
                "syn_correct": int(syn.loc[idx, "correct"]),
                "y_true": int(real.loc[idx, "y_true"]),
                "real_overlay": real.loc[idx, "overlay_path"],
                "syn_overlay": syn.loc[idx, "overlay_path"],
            }
            item.update(sim_metrics(real_cam, syn_cam))
            rows.append(item)

sim_df = pd.DataFrame(rows)
report_dir = ensure_report_dir()
if sim_df.empty:
    print("No comparable CAM pairs found.")
else:
    sim_grouped = sim_df.groupby(["dataset", "group"], as_index=False).agg(
        n=("index", "count"),
        pearson=("pearson", "mean"),
        cosine=("cosine", "mean"),
        top10_iou=("top10_iou", "mean"),
        top10_dice=("top10_dice", "mean"),
        both_correct=("syn_correct", "mean"),
    )
    display(sim_grouped)
    sim_df.to_csv(report_dir / "cam_similarity_to_real_all_rows.csv", index=False)
    sim_grouped.to_csv(report_dir / "cam_similarity_to_real_grouped.csv", index=False)


In [ ]:

if "sim_grouped" in globals() and not sim_grouped.empty:
    for metric in ["pearson", "cosine", "top10_iou", "top10_dice"]:
        pivot = sim_grouped.pivot_table(index="dataset", columns="group", values=metric, aggfunc="first")
        display(pivot)
        ax = pivot.plot(kind="bar", figsize=(9, 4), rot=0)
        ax.set_title(f"Real-vs-synthetic CAM {metric}")
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


In [ ]:

def show_pair_gallery(dataset, group, n=6, sort_by="top10_iou", ascending=True):
    if sim_df.empty:
        print("No similarity rows.")
        return
    sub = sim_df[(sim_df.dataset == dataset) & (sim_df.group == group)].copy()
    if sub.empty:
        print("No rows for", dataset, group)
        return
    sub = sub.sort_values(sort_by, ascending=ascending).head(n)
    fig, axes = plt.subplots(len(sub), 2, figsize=(6, 3 * len(sub)))
    if len(sub) == 1:
        axes = np.array([axes])
    for axrow, (_, row) in zip(axes, sub.iterrows()):
        for ax, key, title in [(axrow[0], "real_overlay", "real"), (axrow[1], "syn_overlay", group)]:
            path = resolve_result_path(row[key])
            if path.exists():
                ax.imshow(Image.open(path))
            ax.set_title(f"{title} idx={row['index']} IoU={row['top10_iou']:.2f}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()

# Example:
# show_pair_gallery("pneumoniamnist", "C_code_default_enhanced", n=6)


## 解读提示

            - 如果某组性能高但 similarity 低，说明它可能走了和 real-trained 不同的捷径。
            - 如果 similarity 高且 Balanced ACC/Macro-F1 也高，说明 synthetic data 更可能保住了真实判别结构。
